## Etapa 2 - Modelare predictiva (Checkpoint 2)

Obiectiv: estimarea tiparului de decizie morala al participantilor folosind predictori demografici, contextuali si psihologici.

Protocol metodologic aplicat:
1. Definirea variabilei tinta cu strategie robusta (extreme split: bottom 35% vs top 35%) pentru reducerea ambiguitatii in etichete.
2. Selectia de caracteristici + feature engineering orientat pe comportament (interactiuni relevante) si preprocesare (imputare, scalare, encodare).
3. Selectie de model si hiperparametri prin cross-validation pe setul de antrenare, cu ranking multi-metric (ROC-AUC + AP).
4. Evaluare finala pe set hold-out (Accuracy, F1, ROC-AUC, Average Precision, Brier score).
5. Analiza extinsa: prag optim de decizie, curbe ROC/PR, importanta variabilelor, robustete pe subgrupuri.

In [ ]:
# Modelare extinsa pentru Checkpoint 2: selectie model + tuning minimal + evaluare hold-out
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score, brier_score_loss
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import pandas as pd
import numpy as np

# 1) Definirea tintei (varianta robusta: excludem zona ambigua din jurul medianei)
saved_rate = pd.to_numeric(df_linked['saved_rate'], errors='coerce')
q_low, q_high = saved_rate.quantile([0.35, 0.65])
extreme_mask = (saved_rate <= q_low) | (saved_rate >= q_high)
df_model = df_linked.loc[extreme_mask].copy()

# Fallback defensiv daca esantionul devine prea mic
if len(df_model) < 5000:
    target_threshold = float(saved_rate.median())
    y = (saved_rate > target_threshold).astype(int)
    df_model = df_linked.copy()
    target_strategy = 'median split (fallback)'
else:
    y = (pd.to_numeric(df_model['saved_rate'], errors='coerce') >= q_high).astype(int)
    target_threshold = float((q_low + q_high) / 2.0)
    target_strategy = 'extreme split (bottom 35% vs top 35%)'

# 2) Definirea predictorilor
feature_cols = [
    'n_responses', 'pedped_rate', 'barrier_rate', 'crossing_signal_mean',
    'default_choice_omission_rate', 'mean_characters', 'mean_diff_characters',
    'age', 'income', 'political', 'religious',
    'scenario_share_age', 'scenario_share_fitness', 'scenario_share_gender',
    'scenario_share_socialvalue', 'scenario_share_species', 'scenario_share_utilitarian', 'scenario_share_random',
    'openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism',
    'template', 'country', 'education', 'gender'
]

X = df_model[feature_cols].copy().replace([np.inf, -np.inf], np.nan)

# 3) Feature engineering orientat pe semnal comportamental + personalitate
X['risk_signal_interaction'] = pd.to_numeric(X['pedped_rate'], errors='coerce') * pd.to_numeric(X['crossing_signal_mean'], errors='coerce')
X['rule_violation_gap'] = pd.to_numeric(X['default_choice_omission_rate'], errors='coerce') - pd.to_numeric(X['crossing_signal_mean'], errors='coerce')
X['personality_stability'] = pd.to_numeric(X['conscientiousness'], errors='coerce') + pd.to_numeric(X['agreeableness'], errors='coerce') - pd.to_numeric(X['neuroticism'], errors='coerce')
X['age_x_utilitarian'] = pd.to_numeric(X['age'], errors='coerce') * pd.to_numeric(X['scenario_share_utilitarian'], errors='coerce')

scenario_cols = [
    'scenario_share_age', 'scenario_share_fitness', 'scenario_share_gender',
    'scenario_share_socialvalue', 'scenario_share_species', 'scenario_share_utilitarian', 'scenario_share_random'
]
scenario_matrix = X[scenario_cols].apply(pd.to_numeric, errors='coerce').clip(lower=1e-6)
scenario_matrix = scenario_matrix.div(scenario_matrix.sum(axis=1), axis=0)
X['scenario_entropy'] = -(scenario_matrix * np.log(scenario_matrix)).sum(axis=1)

categorical_features = ['template', 'country', 'education', 'gender']
numeric_features = [c for c in X.columns if c not in categorical_features]

for col in numeric_features:
    X[col] = pd.to_numeric(X[col], errors='coerce')
    X[col] = X[col].clip(lower=-1e9, upper=1e9)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'f1': 'f1', 'roc_auc': 'roc_auc', 'avg_precision': 'average_precision'}

### Selectie model si evaluare

In celulele urmatoare, modelele candidate sunt comparate prin cross-validation, apoi este ales modelul final si evaluat pe setul de test.

In [ ]:
# 4) Candidati model + evaluare cross-validation
candidates = [
    ('logreg_C0.25', LogisticRegression(max_iter=2000, C=0.25, class_weight='balanced')),
    ('logreg_C0.5', LogisticRegression(max_iter=2000, C=0.5, class_weight='balanced')),
    ('logreg_C1.0', LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced')),
    ('rf_depth10', RandomForestClassifier(n_estimators=600, max_depth=10, min_samples_leaf=3, random_state=42, n_jobs=-1, class_weight='balanced_subsample')),
    ('rf_depth14', RandomForestClassifier(n_estimators=700, max_depth=14, min_samples_leaf=2, random_state=42, n_jobs=-1, class_weight='balanced_subsample')),
    ('extratrees', ExtraTreesClassifier(n_estimators=900, max_depth=None, min_samples_leaf=2, random_state=42, n_jobs=-1, class_weight='balanced_subsample')),
    ('hgb_tuned_a', HistGradientBoostingClassifier(learning_rate=0.03, max_leaf_nodes=63, min_samples_leaf=20, l2_regularization=0.20, random_state=42)),
    ('hgb_tuned_b', HistGradientBoostingClassifier(learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=30, l2_regularization=0.10, random_state=42)),
]

cv_rows = []
fitted_models = {}

for name, model in candidates:
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1, return_train_score=False)
    cv_rows.append({
        'model': name,
        'cv_f1_mean': float(np.mean(scores['test_f1'])),
        'cv_f1_std': float(np.std(scores['test_f1'])),
        'cv_roc_auc_mean': float(np.mean(scores['test_roc_auc'])),
        'cv_roc_auc_std': float(np.std(scores['test_roc_auc'])),
        'cv_ap_mean': float(np.mean(scores['test_avg_precision'])),
        'cv_rank_score': float(0.6 * np.mean(scores['test_roc_auc']) + 0.4 * np.mean(scores['test_avg_precision'])),
    })
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe

cv_results_df = pd.DataFrame(cv_rows).sort_values('cv_rank_score', ascending=False).reset_index(drop=True)
display(cv_results_df)

best_model_name = cv_results_df.loc[0, 'model']
best_model = fitted_models[best_model_name]

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

test_metrics = pd.DataFrame([
    {
        'best_model': best_model_name,
        'accuracy': float(accuracy_score(y_test, y_pred)),
        'f1': float(f1_score(y_test, y_pred)),
        'roc_auc': float(roc_auc_score(y_test, y_proba)),
        'avg_precision': float(average_precision_score(y_test, y_proba)),
        'brier_score': float(brier_score_loss(y_test, y_proba)),
        'class_balance': float(y.mean()),
        'n_samples_modeling': int(len(X)),
    }
])
display(test_metrics)

print('Target strategy:', target_strategy)
print('Target threshold proxy:', round(target_threshold, 4))
print('Selected model:', best_model_name)

In [ ]:
# Analiza extinsa a performantei: ROC/PR, prag optim, importanta variabile, subgrupuri
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix

# 1) Curbe ROC si Precision-Recall
fpr, tpr, roc_thr = roc_curve(y_test, y_proba)
precision, recall, pr_thr = precision_recall_curve(y_test, y_proba)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, label=f'ROC-AUC = {roc_auc_score(y_test, y_proba):.3f}')
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.title('Curba ROC')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(recall, precision, label=f'AP = {average_precision_score(y_test, y_proba):.3f}')
plt.title('Curba Precision-Recall')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend()
plt.tight_layout()
plt.show()

# 2) Prag optim pe F1
threshold_grid = np.linspace(0.2, 0.8, 61)
f1_per_threshold = []
for thr in threshold_grid:
    pred_thr = (y_proba >= thr).astype(int)
    f1_per_threshold.append(f1_score(y_test, pred_thr))

best_thr_idx = int(np.argmax(f1_per_threshold))
best_thr = float(threshold_grid[best_thr_idx])
best_thr_f1 = float(f1_per_threshold[best_thr_idx])

pred_default = (y_proba >= 0.5).astype(int)
pred_opt = (y_proba >= best_thr).astype(int)

cm_default = confusion_matrix(y_test, pred_default)
cm_opt = confusion_matrix(y_test, pred_opt)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.heatmap(cm_default, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix (threshold=0.50)')
plt.xlabel('Predicted')
plt.ylabel('True')

plt.subplot(1, 2, 2)
sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Greens')
plt.title(f'Confusion Matrix (threshold={best_thr:.2f})')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

print(f'F1 @ threshold 0.50: {f1_score(y_test, pred_default):.4f}')
print(f'F1 @ threshold {best_thr:.2f}: {best_thr_f1:.4f}')

In [ ]:
from sklearn.inspection import permutation_importance

# 3) Importanta variabilelor (permutation importance)
perm = permutation_importance(
    best_model, X_test, y_test, n_repeats=5, random_state=42, scoring='roc_auc', n_jobs=-1
)

importance_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=False)

display(importance_df.head(15))

plt.figure(figsize=(10, 6))
top = importance_df.head(12).iloc[::-1]
plt.barh(top['feature'], top['importance_mean'])
plt.title('Top 12 Permutation Importances (ROC-AUC)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
# 4) Analiza pe subgrupuri (acolo unde dimensiunea permite)
if 'gender' in X_test.columns:
    subgroup_rows = []
    temp = X_test.copy()
    temp['y_true'] = y_test.values
    temp['y_proba'] = y_proba

    for g, dfg in temp.groupby('gender', dropna=False):
        if len(dfg) >= 200 and dfg['y_true'].nunique() == 2:
            subgroup_rows.append({
                'gender': str(g),
                'n': int(len(dfg)),
                'roc_auc': float(roc_auc_score(dfg['y_true'], dfg['y_proba'])),
                'avg_precision': float(average_precision_score(dfg['y_true'], dfg['y_proba']))
            })

    if len(subgroup_rows) > 0:
        subgroup_df = pd.DataFrame(subgroup_rows).sort_values('n', ascending=False)
        display(subgroup_df)
    else:
        print('Analiza pe subgrupuri: esantioane insuficiente pentru estimari stabile.')